---
# 🏁 Part 2 — Race Outcome Prediction (LSTM)

Each driver's race is represented as a **sequence of laps** → the LSTM predicts their **finishing position bucket**.

```
[lap1_features] → [lap2_features] → ... → [lapN_features]  ──►  finishing bucket
```

**Buckets:** P1-P4 · P5-P8 · P9-P12 · P13-P16 · P17-P20  
**Architecture:** Bidirectional 2-layer LSTM → Dense → 5-class softmax  
**Framework:** PyTorch

> ⚠️ **Prerequisite:** Run `f1_lap_prediction.ipynb` first to generate `encoders.pkl`, `xgb_model.pkl`, and `laps_raw.csv`.


## Cell A — Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import fastf1
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight

# ── Shared config ──────────────────────────────────────────────────────────────
CACHE_DIR   = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

YEAR        = 2024
ROUNDS      = list(range(1, 25))   # Full 2024 season (24 rounds)
MIN_LAPS    = 3
MIN_LAP_SEC = 60
MAX_LAP_SEC = 130
N_BUCKETS   = 5

BUCKET_LABELS = ["P1–P4", "P5–P8", "P9–P12", "P13–P16", "P17–P20"]

def position_to_bucket(pos):
    """Map finishing position (1-20) to bucket index (0-4)."""
    return (pos - 1) // 4

# ── Load artifacts from Notebook 1 ────────────────────────────────────────────
encoders  = joblib.load("encoders.pkl")
xgb_model = joblib.load("xgb_model.pkl")
print("✅ Loaded encoders and XGBoost model from Notebook 1")
print(f"   Encoder classes — Compound: {list(encoders['Compound'].classes_)}")


In [ ]:
# Uncomment if torch is not installed
# !pip install torch seaborn

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split as tts
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch {torch.__version__} | Device: {DEVICE}")


## Cell B — Collect & Build Feature Sequences

11 features per lap:

| # | Feature | Notes |
|---|---------|-------|
| 1 | `LapNumber` | Lap count |
| 2 | `LapTimeNorm` | (lap_time − race_median) / race_median |
| 3 | `TyreLife` | Laps on current tyre |
| 4 | `Compound_enc` | From Notebook 1 encoder |
| 5 | `IsPit` | Binary pit-stop flag |
| 6 | `TrackPos` | Current race position this lap |
| 7 | `Driver_enc` | From Notebook 1 encoder (constant per sequence) |
| 8 | `Team_enc` | From Notebook 1 encoder (constant per sequence) |
| 9 | `GridPosition` | Starting grid slot (constant per sequence) |
| 10 | `XGB_PredNorm` | XGBoost predicted lap time, normalised |
| 11 | `XGB_Residual` | LapTimeNorm − XGB_PredNorm (pace deviation) |


In [ ]:
def safe_encode(col, val):
    """Encode a value using Notebook 1's LabelEncoder; fall back to median index for unseen labels."""
    enc = encoders[col]
    if val in enc.classes_:
        return int(enc.transform([val])[0])
    return len(enc.classes_) // 2


def collect_sequence_data(year, rounds):
    all_sequences = []

    for rnd in rounds:
        try:
            session = fastf1.get_session(year, rnd, "R")
            session.load(laps=True, telemetry=False, weather=True, messages=False)
            laps = session.laps.copy()

            # ── Round-level constants ────────────────────────────────────────────
            event_name = session.event["EventName"]
            weather    = session.weather_data
            avg_track  = weather["TrackTemp"].mean() if weather is not None and not weather.empty else 40.0
            avg_air    = weather["AirTemp"].mean()   if weather is not None and not weather.empty else 28.0
            event_enc  = safe_encode("EventName", event_name)

            # ── Finishing & grid positions ───────────────────────────────────────
            results = session.results[["Abbreviation", "Position", "GridPosition"]].copy()
            results.columns = ["Driver", "FinishPosition", "GridPosition"]
            results["FinishPosition"] = pd.to_numeric(results["FinishPosition"], errors="coerce")
            results["GridPosition"]   = pd.to_numeric(results["GridPosition"],   errors="coerce").fillna(20)
            results = results.dropna(subset=["FinishPosition"])
            results["FinishPosition"] = results["FinishPosition"].astype(int)
            results["GridPosition"]   = results["GridPosition"].astype(int)

            # ── Lap-level derived columns ────────────────────────────────────────
            laps["LapTimeSec"] = laps["LapTime"].apply(
                lambda x: x.total_seconds() if pd.notna(x) else np.nan
            )
            laps["IsPit"]    = laps["PitOutTime"].notna().astype(int)
            laps["TyreLife"] = laps["TyreLife"].fillna(0)
            laps["TrackPos"] = pd.to_numeric(laps["Position"], errors="coerce").fillna(20)

            # Normalise lap time within this race
            median_lt = laps["LapTimeSec"].median()
            laps["LapTimeNorm"] = (laps["LapTimeSec"] - median_lt) / (median_lt + 1e-6)

            total_laps = max(laps["LapNumber"].max(), 1)

            # ── Per-driver sequences ─────────────────────────────────────────────
            for driver, grp in laps.groupby("Driver"):
                pos_row = results[results["Driver"] == driver]
                if pos_row.empty:
                    continue
                finish_pos = int(pos_row["FinishPosition"].values[0])
                grid_pos   = int(pos_row["GridPosition"].values[0])
                if finish_pos < 1 or finish_pos > 20:
                    continue

                team = grp["Team"].mode()[0] if "Team" in grp.columns and not grp["Team"].isna().all() else "Unknown"

                driver_laps = grp.sort_values("LapNumber").copy()
                driver_laps = driver_laps.fillna(driver_laps.median(numeric_only=True))

                if len(driver_laps) < 5:
                    continue

                n = len(driver_laps)
                lap_nums   = driver_laps["LapNumber"].values
                tyre_lives = driver_laps["TyreLife"].values
                compound_enc_vec = driver_laps["Compound"].fillna("UNKNOWN").apply(
                    lambda c: safe_encode("Compound", c)
                ).values
                driver_enc = safe_encode("Driver", driver)
                team_enc   = safe_encode("Team", team)

                # XGBoost features → predicted lap times
                xgb_feats = pd.DataFrame({
                    "LapNumber"     : lap_nums,
                    "LapFrac"       : lap_nums / total_laps,
                    "TyreLife"      : tyre_lives,
                    "TyreLife2"     : tyre_lives ** 2,
                    "Compound_enc"  : compound_enc_vec,
                    "Driver_enc"    : np.full(n, driver_enc),
                    "Team_enc"      : np.full(n, team_enc),
                    "EventName_enc" : np.full(n, event_enc),
                    "TrackTemp"     : np.full(n, avg_track),
                    "AirTemp"       : np.full(n, avg_air),
                })
                xgb_pred      = xgb_model.predict(xgb_feats)
                xgb_pred_norm = (xgb_pred - median_lt) / (median_lt + 1e-6)
                xgb_residual  = driver_laps["LapTimeNorm"].values - xgb_pred_norm

                lap_matrix = np.column_stack([
                    lap_nums,                              # 1. LapNumber
                    driver_laps["LapTimeNorm"].values,     # 2. LapTimeNorm
                    tyre_lives,                            # 3. TyreLife
                    compound_enc_vec,                      # 4. Compound_enc
                    driver_laps["IsPit"].values,           # 5. IsPit
                    driver_laps["TrackPos"].values,        # 6. TrackPos
                    np.full(n, driver_enc),                # 7. Driver_enc  (constant)
                    np.full(n, team_enc),                  # 8. Team_enc    (constant)
                    np.full(n, grid_pos),                  # 9. GridPosition (constant)
                    xgb_pred_norm,                         # 10. XGB predicted lap time
                    xgb_residual,                          # 11. XGB residual
                ]).astype(np.float32)

                all_sequences.append({
                    "round"           : rnd,
                    "driver"          : driver,
                    "finish_position" : finish_pos,
                    "bucket"          : position_to_bucket(finish_pos),
                    "laps"            : lap_matrix,
                })

            print(f"  Round {rnd:>2} ✓  {event_name:<30} — sequences so far: {len(all_sequences)}")

        except Exception as e:
            print(f"  Round {rnd:>2} ✗ skipped ({e})")

    return all_sequences


print("Collecting sequence data (uses FastF1 cache — fast after first run)...\n")
sequences = collect_sequence_data(YEAR, ROUNDS)
print(f"\n📦 Total driver-race sequences : {len(sequences)}")
print(f"   Features per lap            : {sequences[0]['laps'].shape[1]}")
print(f"   Example — {sequences[0]['driver']} Round {sequences[0]['round']}: "
      f"{sequences[0]['laps'].shape[0]} laps → P{sequences[0]['finish_position']} "
      f"(bucket {sequences[0]['bucket']}: {BUCKET_LABELS[sequences[0]['bucket']]})")


## Cell C — Dataset & DataLoader

In [ ]:
# ── Normalise features across all sequences ──────────────────────────────────
all_laps_flat = np.vstack([s["laps"] for s in sequences])
scaler = StandardScaler()
scaler.fit(all_laps_flat)

for s in sequences:
    s["laps"] = scaler.transform(s["laps"]).astype(np.float32)

N_FEATURES = sequences[0]["laps"].shape[1]
print(f"Features per lap after scaling: {N_FEATURES}")


# ── PyTorch Dataset ───────────────────────────────────────────────────────────
class RaceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        s = self.sequences[idx]
        x = torch.tensor(s["laps"], dtype=torch.float32)        # (n_laps, N_FEATURES)
        y = torch.tensor(s["bucket"], dtype=torch.long)         # 0-indexed bucket
        return x, y


def collate_fn(batch):
    xs, ys = zip(*batch)
    xs_padded = pad_sequence(xs, batch_first=True, padding_value=0.0)
    return xs_padded, torch.stack(ys)


# ── Train / Val split (80/20) ─────────────────────────────────────────────────
train_seqs, val_seqs = tts(sequences, test_size=0.2, random_state=42)

train_ds = RaceDataset(train_seqs)
val_ds   = RaceDataset(val_seqs)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, collate_fn=collate_fn)

# ── Class weights to handle bucket imbalance ──────────────────────────────────
train_labels = [s["bucket"] for s in train_seqs]
class_weights = compute_class_weight("balanced", classes=np.arange(N_BUCKETS), y=train_labels)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"Class weights: { {BUCKET_LABELS[i]: round(w, 3) for i, w in enumerate(class_weights)} }")

print(f"\nTrain sequences : {len(train_ds)}")
print(f"Val sequences   : {len(val_ds)}")
print(f"Input shape (sample batch): {next(iter(train_loader))[0].shape}")


## Cell D — Bidirectional LSTM Model

```
Input  (batch, max_laps, 11 features)
  │
  ▼
BiLSTM layer 1  (hidden=64 each direction → 128 output, dropout=0.3)
  │
BiLSTM layer 2  (hidden=32 each direction → 64 output)
  │  (take last timestep: forward-last ‖ backward-last)
  ▼
LayerNorm(64) → Dropout(0.3)
  │
Linear(64 → 5)
  │
Softmax → P(P1-P4), P(P5-P8), P(P9-P12), P(P13-P16), P(P17-P20)
```

Bidirectional LSTM lets the model see both the race progression forward **and** the final laps (which carry the strongest finishing-position signal) directly.


In [ ]:
class RaceLSTM(nn.Module):
    def __init__(self, input_size, hidden1=64, hidden2=32,
                 n_classes=N_BUCKETS, dropout=0.3):
        super().__init__()

        self.lstm1 = nn.LSTM(
            input_size, hidden1,
            batch_first=True, bidirectional=True, dropout=dropout
        )
        self.lstm2 = nn.LSTM(
            hidden1 * 2, hidden2,       # input is 2*hidden1 due to bidirectional
            batch_first=True, bidirectional=True
        )
        self.norm    = nn.LayerNorm(hidden2 * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden2 * 2, n_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, _ = self.lstm1(x)           # (batch, seq_len, 2*hidden1)
        out, _ = self.lstm2(out)         # (batch, seq_len, 2*hidden2)
        out    = out[:, -1, :]           # last timestep → (batch, 2*hidden2)
        out    = self.norm(out)
        out    = self.dropout(out)
        return self.fc(out)              # (batch, n_classes) — raw logits


model = RaceLSTM(input_size=N_FEATURES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")


## Cell E — Training Loop

In [ ]:
EPOCHS    = 100
LR        = 1e-3

criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "val_loss": [], "val_top1": [], "val_top2": []}


def accuracy_topk(logits, targets, k=1):
    topk = logits.topk(k, dim=1).indices
    correct = topk.eq(targets.view(-1, 1).expand_as(topk))
    return correct.any(dim=1).float().mean().item()


print(f"Training for {EPOCHS} epochs on {DEVICE} | {N_BUCKETS} buckets | {N_FEATURES} features\n")
print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Val Loss':>9}  {'Top-1 Acc':>10}  {'Top-2 Acc':>10}")
print("-" * 60)

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_ds)
    scheduler.step()

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    val_loss, top1, top2 = 0.0, 0.0, 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            val_loss += criterion(logits, yb).item() * len(xb)
            top1     += accuracy_topk(logits, yb, k=1) * len(xb)
            top2     += accuracy_topk(logits, yb, k=2) * len(xb)
    val_loss /= len(val_ds)
    top1      /= len(val_ds)
    top2      /= len(val_ds)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_top1"].append(top1)
    history["val_top2"].append(top2)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "lstm_best.pt")
        tag = " ← best"
    else:
        tag = ""

    if epoch % 10 == 0 or epoch == 1:
        print(f"{epoch:>6}  {train_loss:>11.4f}  {val_loss:>9.4f}  "
              f"{top1*100:>9.1f}%  {top2*100:>9.1f}%{tag}")

print(f"\n✅ Training complete. Best val loss: {best_val_loss:.4f}")
print("   Model saved → lstm_best.pt")


## Cell F — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), facecolor="#0f0f0f")
fig.suptitle("LSTM Training History — Race Outcome Prediction (5 Buckets)",
             color="white", fontsize=14)

epochs_range = range(1, EPOCHS + 1)

ax = axes[0]
ax.plot(epochs_range, history["train_loss"], color="#e8c84a", label="Train")
ax.plot(epochs_range, history["val_loss"],   color="#4ae8a0", label="Val", linestyle="--")
ax.set_title("Loss (CrossEntropy)", color="white")
ax.set_xlabel("Epoch", color="#aaa")
ax.legend(facecolor="#222", labelcolor="white", fontsize=9)

ax = axes[1]
ax.plot(epochs_range, [v * 100 for v in history["val_top1"]],
        color="#e84a4a", label="Top-1")
ax.axhline(20, color="#555", linestyle=":", linewidth=1, label="Random baseline (20%)")
ax.set_title("Top-1 Accuracy (%)", color="white")
ax.set_xlabel("Epoch", color="#aaa")
ax.set_ylabel("%", color="#aaa")
ax.legend(facecolor="#222", labelcolor="white", fontsize=9)

ax = axes[2]
ax.plot(epochs_range, [v * 100 for v in history["val_top2"]],
        color="#4a9fe8", label="Top-2")
ax.axhline(40, color="#555", linestyle=":", linewidth=1, label="Random baseline (40%)")
ax.set_title("Top-2 Accuracy (%)", color="white")
ax.set_xlabel("Epoch", color="#aaa")
ax.set_ylabel("%", color="#aaa")
ax.legend(facecolor="#222", labelcolor="white", fontsize=9)

for ax in axes:
    ax.set_facecolor("#1a1a1a")
    ax.tick_params(colors="#aaa")
    for s in ax.spines.values():
        s.set_edgecolor("#333")

plt.tight_layout()
plt.savefig("lstm_training.png", dpi=150, bbox_inches="tight", facecolor="#0f0f0f")
plt.show()
print("✅ Saved → lstm_training.png")


## Cell G — Confusion Matrix & Final Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load("lstm_best.pt", map_location=DEVICE))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(DEVICE))
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(yb.numpy())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)

top1_final = (all_preds == all_targets).mean() * 100

top2_correct = 0
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(DEVICE))
        top2   = logits.topk(2, dim=1).indices.cpu().numpy()
        for i, t in enumerate(yb.numpy()):
            if t in top2[i]:
                top2_correct += 1
top2_final = top2_correct / len(val_ds) * 100

print(f"{'='*42}")
print(f"  Final Evaluation (Best Checkpoint)")
print(f"{'='*42}")
print(f"  Top-1 Accuracy : {top1_final:.1f}%  (random = 20%)")
print(f"  Top-2 Accuracy : {top2_final:.1f}%  (random = 40%)")
print(f"{'='*42}")

# ── 5×5 Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(all_targets, all_preds, labels=list(range(N_BUCKETS)))

fig, ax = plt.subplots(figsize=(8, 6), facecolor="#0f0f0f")
sns.heatmap(
    cm, annot=True, fmt="d", cmap="YlOrRd",
    xticklabels=BUCKET_LABELS,
    yticklabels=BUCKET_LABELS,
    ax=ax, linewidths=0.5, linecolor="#333"
)
ax.set_title("Confusion Matrix — Predicted vs Actual Position Bucket",
             color="white", fontsize=13, pad=12)
ax.set_xlabel("Predicted Bucket", color="#aaa")
ax.set_ylabel("Actual Bucket",    color="#aaa")
ax.tick_params(colors="white")
ax.set_facecolor("#1a1a1a")
fig.patch.set_facecolor("#0f0f0f")

plt.tight_layout()
plt.savefig("lstm_confusion.png", dpi=150, bbox_inches="tight", facecolor="#0f0f0f")
plt.show()
print("✅ Saved → lstm_confusion.png")


## Cell H — Predict a Race Outcome

Feed in any driver's lap data (from the loaded sequences) and get a predicted finishing bucket.


In [ ]:
def predict_finish_position(driver, round_number, top_k=3):
    match = [
        s for s in sequences
        if s["driver"] == driver and s["round"] == round_number
    ]
    if not match:
        print(f"No data found for {driver} in round {round_number}")
        return

    seq = match[0]
    x   = torch.tensor(seq["laps"], dtype=torch.float32).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    top_buckets = np.argsort(probs)[::-1][:top_k]
    actual_bucket = position_to_bucket(seq["finish_position"])

    print(f"\n🏎️  Driver: {driver} | Round: {round_number}")
    print(f"   Actual finish : P{seq['finish_position']}  →  {BUCKET_LABELS[actual_bucket]}")
    print(f"   Predicted (top {top_k}):")
    for rank, b in enumerate(top_buckets, 1):
        hit = " ✓" if b == actual_bucket else ""
        print(f"     {rank}. {BUCKET_LABELS[b]:<10}  ({probs[b]*100:.1f}%){hit}")


# ── Try it ────────────────────────────────────────────────────────────────────
predict_finish_position(driver="VER", round_number=5)
predict_finish_position(driver="HAM", round_number=5)
